In [1]:
import fitz          # PyMuPDF
import pdfplumber
import json
import os
import io
from pathlib import Path
from PIL import Image

# Verify versions
print(f"PyMuPDF version  : {fitz.version}")
print(f"pdfplumber       : {pdfplumber.__version__}")
print("✅ Core imports OK")

PyMuPDF version  : ('1.26.7', '1.26.12', None)
pdfplumber       : 0.11.9
✅ Core imports OK


In [2]:
# petition path
# PDF_PATH = r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\jusnl-2025.pdf"
PDF_PATH = r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf"
# PDF_PATH = r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\jharkhand_regulation.pdf"
# This is what we discussed — document-level metadata
DOC_METADATA = {
    "utility": "JUSNL",
    "document_type": "petition",   # petition / tariff_order / regulation
    "filing_year": "FY26",         # the year the petition covers
    "regulator": "JSERC",
    "source_file": PDF_PATH
}

In [3]:
# Check file exists
if Path(PDF_PATH).exists():
    print(f"✅ PDF found: {PDF_PATH}")
    doc = fitz.open(PDF_PATH)
    print(f"   Total pages: {len(doc)}")
    doc.close()
else:
    print(f"⚠️  PDF not found at: {PDF_PATH}")
    print("   Place your PDF in the data/petitions/ folder and update PDF_PATH above")

✅ PDF found: D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf
   Total pages: 66


In [4]:
TOC_KEYWORDS = [
    "table of contents",
    "contents",
    "index",

]

INVALID_TOC_TITLES = [
    r"list of tables..",
    r"list of figures"
]

with pdfplumber.open(PDF_PATH) as pdf:

    for i , page in enumerate(pdf.pages[:4]):
        text = page.extract_text() or ""
        text_lower = text.lower()
        # print(f"--- Page {i+1} ---")
        print(text[-100:])  # print first 200 chars
        if any(keyword for keyword in TOC_KEYWORDS if keyword in text_lower):
            print(f"   ✅ TOC detected on the page {i+1}")
        else:
            print(f"   No TOC keywords found on the page {i+1}")



MISSION, RANCHI
SUBMITTED BY:
JHARKHAND URJA SANCHARAN NIGAM LIMITED,
KUSAI COLONY, RANCHI – 834 010
   No TOC keywords found on the page 1
itioner is a
Company constituted under the provisions of Government of Jharkhand, General
2 | P a ge
   No TOC keywords found on the page 2
Electricity
Regulatory Commission (Terms and Conditions for Determination of Transmission
3 | P a ge
   No TOC keywords found on the page 3
khand Urja Sancharan Nigam Limited
(Petitioner)
Authorized Signatory
Place: Ranchi
Dated:
4 | P a ge
   No TOC keywords found on the page 4


In [5]:
import re
import pdfplumber
TOC_KEYWORDS = [
    "table of contents",
    "contents",
    "index",

]

INVALID_TOC_TITLES = [
    r"list of tables..",
    r"list of figures"
]


TOC_LINE_PATTERN = r".+\.{2,}\s*\d+$"

LIST_OF_TABLE_PATTERN=r"^table\s+\d+"


def detect_toc_pages(pdf_path, max_pages=10):

    toc_pages = []

    toc_started = False

    with pdfplumber.open(pdf_path) as pdf:

        for i, page in enumerate(pdf.pages[1:max_pages]):

            text = page.extract_text() or ""

            cleaned = re.sub(r"\s+", " ", text.lower())


            has_valid_toc_title=any(keyword for keyword in TOC_KEYWORDS if keyword in cleaned)

            invalid_keyword=any(keyword for keyword in INVALID_TOC_TITLES if keyword in cleaned)




            if any(keyword for keyword in INVALID_TOC_TITLES if keyword in cleaned):
                print(f"   ✅ Invalid TOC detected on the page {i+1}")


            lines = text.splitlines()
            toc_like_lines = sum(
                1
                for line in lines
                if re.match(TOC_LINE_PATTERN, line.strip())
            )


            list_of_table_pattern =sum(1 for line in lines if  re.match(r"table\s+\d+", line.lower().strip()))


            # print(toc_like_lines, has_valid_toc_title, list_of_table_pattern, invalid_keyword, i+2)

            if (toc_like_lines>10 or (has_valid_toc_title )) and (list_of_table_pattern==0) :

                    toc_pages.append(i+2)
                    # con


    return toc_pages

toc_pages = detect_toc_pages(PDF_PATH)
print(toc_pages)
# print(f"TOC detected on pages: {', '.join(str(p+1) for p in toc_pages)}")

[5, 6]


In [6]:
import re
import pdfplumber


TOC_KEYWORDS = [
    "table of contents",
    "contents",
    "index"
]

INVALID_TOC_TITLES = [
    "list of tables",
    "list of figures"
]

TOC_LINE_PATTERN = r".+\.{2,}\s*\d+$"

LIST_OF_TABLE_PATTERN = r"^table\s+\d+"


def detect_toc_pages(pdf_path, max_pages=10):

    toc_pages = []

    with pdfplumber.open(pdf_path) as pdf:

        for i, page in enumerate(pdf.pages[:max_pages]):

            text = page.extract_text() or ""

            cleaned = re.sub(r"\s+", " ", text.lower())

            # valid TOC title
            has_valid_toc_title = any(
                keyword in cleaned
                for keyword in TOC_KEYWORDS
            )

            # invalid TOC title
            has_invalid_title = any(
                keyword in cleaned
                for keyword in INVALID_TOC_TITLES
            )

            lines = text.splitlines()

            # dotted TOC-like lines
            toc_like_lines = sum(
                1
                for line in lines
                if re.match(
                    TOC_LINE_PATTERN,
                    line.strip()
                )
            )

            # table-index style lines
            table_lines = sum(
                1
                for line in lines
                if re.match(
                    LIST_OF_TABLE_PATTERN,
                    line.lower().strip()
                )
            )

            # FINAL LOGIC
            is_toc_page = (
                (
                    has_valid_toc_title
                    or toc_like_lines > 10
                )
                and not has_invalid_title
                and table_lines < 3
            )

            if is_toc_page:

                print(
                    f"✅ TOC detected on page {i+1}"
                )

                toc_pages.append(i + 1)

    return toc_pages

In [7]:
toc_pages = detect_toc_pages(PDF_PATH)
print(toc_pages)

✅ TOC detected on page 5
✅ TOC detected on page 6
[5, 6]


In [8]:

with pdfplumber.open(PDF_PATH) as pdf:

    for page_num in toc_pages:

        page = pdf.pages[page_num - 1]

        text = page.extract_text() or ""

        lines = text.splitlines()

        print(lines)

['ARR and Tariff Petition for FY 2025-26 JUSNL', 'Contents', '1. Introduction .......................................................................................................................................................... 8', '1.1. Background ....................................................................................................................................... 8', '1.2. Profile of JUSNL ............................................................................................................................... 9', '1.3. Procedural History .......................................................................................................................... 11', '1.4. Rationale for filing of Instant Petition ............................................................................................. 12', '1.5. Contents of the Petition ................................................................................................................... 12', 

In [9]:
import pdfplumber


import pdfplumber


def extract_clean_page_text(page):

    text_lines = page.extract_text_lines()

    height = page.height

    top_threshold = height * 0.08

    bottom_threshold = height * 0.92

    cleaned_lines = []

    for line in text_lines:

        # line coordinates
        top = line["top"]
        bottom = line["bottom"]

        # remove header
        if top < top_threshold:
            continue

        # remove footer
        if bottom > bottom_threshold:
            continue

        cleaned_lines.append(
            line["text"]
        )

    return "\n".join(cleaned_lines)
import re

def detect_table_lines(text):

    lines = text.splitlines()

    table_lines = []

    for line in lines:

        # count numbers
        numbers = re.findall(
            r"[-+]?\d[\d,]*\.?\d*",
            line
        )

        # likely table row
        if len(numbers) >= 2:

            table_lines.append(line)

    return table_lines


In [10]:
with pdfplumber.open(PDF_PATH) as pdf:
    res = ""
    for i in toc_pages:
        page = pdf.pages[i-1]

        clean_text = extract_clean_page_text(page)

        res += clean_text+"\n"
    print(res)

    op=detect_table_lines(res)

Contents
1. Introduction .......................................................................................................................................................... 8
1.1. Background ....................................................................................................................................... 8
1.2. Profile of JUSNL ............................................................................................................................... 9
1.3. Procedural History .......................................................................................................................... 11
1.4. Rationale for filing of Instant Petition ............................................................................................. 12
1.5. Contents of the Petition ................................................................................................................... 12
2. Overall Approach and Provision of Law .............................

In [11]:

op


['1. Introduction .......................................................................................................................................................... 8',
 '1.1. Background ....................................................................................................................................... 8',
 '1.2. Profile of JUSNL ............................................................................................................................... 9',
 '1.3. Procedural History .......................................................................................................................... 11',
 '1.4. Rationale for filing of Instant Petition ............................................................................................. 12',
 '1.5. Contents of the Petition ................................................................................................................... 12',
 '2. Overall Approach and Provision of Law ............

In [12]:
import re


TOC_PATTERN = (
    r'^(\d+(?:\.\d+)*)\.?\s+'
    r'(.+?)'
    r'\.{2,}\s*'
    r'(\d+)$'
)


def parse_toc_lines(lines):

    toc_entries = []

    for line in lines:

        line = line.strip()

        match = re.match(
            TOC_PATTERN,
            line
        )

        if match:

            section_num = match.group(1)

            title = match.group(2).strip()

            page_no = int(match.group(3))

            toc_entries.append({

                "section_num":
                    section_num,

                "title":
                    title,

                "page":
                    page_no,

                "full_heading":
                    f"{section_num} {title}"
            })

    return toc_entries

In [13]:
toc_entries = parse_toc_lines(op)

for entry in toc_entries[:]:

    print(entry)


{'section_num': '1', 'title': 'Introduction', 'page': 8, 'full_heading': '1 Introduction'}
{'section_num': '1.1', 'title': 'Background', 'page': 8, 'full_heading': '1.1 Background'}
{'section_num': '1.2', 'title': 'Profile of JUSNL', 'page': 9, 'full_heading': '1.2 Profile of JUSNL'}
{'section_num': '1.3', 'title': 'Procedural History', 'page': 11, 'full_heading': '1.3 Procedural History'}
{'section_num': '1.4', 'title': 'Rationale for filing of Instant Petition', 'page': 12, 'full_heading': '1.4 Rationale for filing of Instant Petition'}
{'section_num': '1.5', 'title': 'Contents of the Petition', 'page': 12, 'full_heading': '1.5 Contents of the Petition'}
{'section_num': '2', 'title': 'Overall Approach and Provision of Law', 'page': 13, 'full_heading': '2 Overall Approach and Provision of Law'}
{'section_num': '2.1', 'title': 'Present Approach', 'page': 13, 'full_heading': '2.1 Present Approach'}
{'section_num': '2.2', 'title': 'Data and information sources', 'page': 13, 'full_heading

In [14]:
def enrich_toc_hierarchy(toc_entries):

    current_main_section = None

    for entry in toc_entries:

        section_num = entry["section_num"]

        # top-level section
        if "." not in section_num:

            current_main_section = (
                entry["full_heading"]
            )

            entry["main_section"] = (
                current_main_section
            )

        else:

            entry["main_section"] = (
                current_main_section
            )

    return toc_entries

In [15]:
toc_entries = enrich_toc_hierarchy(
    toc_entries
)

In [16]:
toc_entries

[{'section_num': '1',
  'title': 'Introduction',
  'page': 8,
  'full_heading': '1 Introduction',
  'main_section': '1 Introduction'},
 {'section_num': '1.1',
  'title': 'Background',
  'page': 8,
  'full_heading': '1.1 Background',
  'main_section': '1 Introduction'},
 {'section_num': '1.2',
  'title': 'Profile of JUSNL',
  'page': 9,
  'full_heading': '1.2 Profile of JUSNL',
  'main_section': '1 Introduction'},
 {'section_num': '1.3',
  'title': 'Procedural History',
  'page': 11,
  'full_heading': '1.3 Procedural History',
  'main_section': '1 Introduction'},
 {'section_num': '1.4',
  'title': 'Rationale for filing of Instant Petition',
  'page': 12,
  'full_heading': '1.4 Rationale for filing of Instant Petition',
  'main_section': '1 Introduction'},
 {'section_num': '1.5',
  'title': 'Contents of the Petition',
  'page': 12,
  'full_heading': '1.5 Contents of the Petition',
  'main_section': '1 Introduction'},
 {'section_num': '2',
  'title': 'Overall Approach and Provision of Law

In [17]:
def is_leaf_section(toc_entries, idx):

    current = toc_entries[idx]["section_num"]

    for j in range(idx + 1, len(toc_entries)):

        next_sec = toc_entries[j]["section_num"]

        # child detected
        if next_sec.startswith(current + "."):
            return False

        # moved outside hierarchy
        if not next_sec.startswith(current):
            break

    return True

def get_level(section_num):

    return len(section_num.split("."))


def build_section_ranges(toc_entries, total_pages):

    ranges = []

    for i, entry in enumerate(toc_entries):

        current = entry["section_num"]
        current_level = get_level(current)

        start_page = entry["page"]

        end_page = total_pages

        # search for next SAME LEVEL section
        for j in range(i + 1, len(toc_entries)):

            next_entry = toc_entries[j]

            next_section = next_entry["section_num"]

            next_level = get_level(next_section)

            # same hierarchy level
            if next_level == current_level:

                end_page = next_entry["page"] - 1
                break

        ranges.append({

            "section_num": current,
            "title": entry["title"],
            "page_start": start_page,
            "page_end": end_page
        })

    return ranges

import pdfplumber

with pdfplumber.open(PDF_PATH) as pdf:

    total_pages = len(pdf.pages)

section_ranges=build_section_ranges(toc_entries,total_pages )

In [ ]:
hierarchy

{'level_1': '6 Determination of Transmission Tariff for FY 2025-26',
 'level_2': '6.1 Preamble'}

In [ ]:
import pdfplumber


def extract_clean_page_text(page):

    words = page.extract_words()

    height = page.height

    top_threshold = height * 0.08

    bottom_threshold = height * 0.92

    filtered_words = []

    for word in words:

        # remove header
        if word["top"] < top_threshold:
            continue

        # remove footer
        if word["bottom"] > bottom_threshold:
            continue

        filtered_words.append(word)

    # rebuild lines
    lines = {}

    for word in filtered_words:

        top = round(word["top"], 1)

        if top not in lines:
            lines[top] = []

        lines[top].append(word["text"])

    final_lines = []

    for top in sorted(lines.keys()):

        line = " ".join(lines[top])

        final_lines.append(line)

    return "\n".join(final_lines)



def extract_section_chunks(
        pdf_path,
        section_ranges
):

    chunks = []

    with pdfplumber.open(pdf_path) as pdf:

        for section in section_ranges:

            section_text = ""

            start = section["page_start"]
            end = section["page_end"]

            # collect all pages
            for pg in range(start, end + 1):

                page = pdf.pages[pg - 1]

                # CLEAN TEXT HERE
                page_text = extract_clean_page_text(page)

                section_text += "\n" + page_text

            chunks.append({

                **section,

                "chunk_type":
                    "text",

                "text":
                    section_text.strip()
            })

    return chunks


In [ ]:
chunks = extract_section_chunks(
    PDF_PATH,
    section_ranges
)

print(chunks[0])

{'section_num': '1', 'title': 'Introduction', 'page_start': 8, 'page_end': 12, 'chunk_type': 'text', 'text': '1. Introduction'}


In [ ]:
chunks

[{'section_num': '1',
  'title': 'Introduction',
  'page_start': 8,
  'page_end': 12,
  'chunk_type': 'text',
  'text': '1. Introduction'},
 {'section_num': '1.1',
  'title': 'Background',
  'page_start': 8,
  'page_end': 8,
  'chunk_type': 'text',
  'text': '1.1. Background\n1.1.1. The\nerstwhile Jharkhand State Electricity Board (“Board” or “JSEB”) was a statutory\nbody constituted under Section 5 of the Electricity (Supply) Act, 1948 and was\nengaged in electricity generation, transmission, distribution and related activities in\nthe State of Jharkhand. The erstwhile Jharkhand State Electricity Board (JSEB)\nwas constituted on March 10, 2001 under the Electricity (Supply) Act, 1948 as a\nresult of the bifurcation of the erstwhile State of Bihar. Before that, the Jharkhand\nState Electricity Board (JSEB) was the predominant entity entrusted with the task of\ngenerating, transmitting and supplying power in the State.\n1.1.2. Jharkhand Urja Vikas Nigam Ltd. (herein after\nto be referre

## Table Handling 

In [ ]:
import camelot


def dataframe_to_text(df):

    rows = df.values.tolist()

    headers = rows[0]

    data_rows = rows[1:]

    table_lines = []

    for row in data_rows:

        pairs = []

        for h, v in zip(headers, row):

            pairs.append(f"{h}: {v}")

        table_lines.append(
            " | ".join(pairs)
        )

    return "\n".join(table_lines)



def extract_table_chunks(
        pdf_path,
        section_ranges
):

    table_chunks = []

    for section in section_ranges:

        start = section["page_start"]
        end = section["page_end"]

        for pg in range(start, end + 1):

            try:

                tables = camelot.read_pdf(
                    pdf_path,
                    pages=str(pg),
                    flavor="stream"
                )

                for i, table in enumerate(tables):

                    df = table.df

                    table_text = dataframe_to_text(df)

                    table_chunks.append({

                        **section,

                        "page_number": pg,

                        "table_index": i,

                        "chunk_type": "table",

                        "text": table_text
                    })

            except Exception as e:

                print(f"Error page {pg}: {e}")

    return table_chunks

In [ ]:
def extract_table_chunks(
        pdf_path,
        section_ranges
):

    table_chunks = []

    for section in section_ranges:

        start = section["page_start"]
        end = section["page_end"]

        for pg in range(start, end + 1):

            try:

                tables = camelot.read_pdf(
                    pdf_path,
                    pages=str(pg),
                    flavor="stream"
                )

                for i, table in enumerate(tables):

                    df = table.df

                    table_text = dataframe_to_text(df)

                    table_chunks.append({

                        **section,

                        "page_number": pg,

                        "table_index": i,

                        "chunk_type": "table",

                        "text": table_text
                    })

            except Exception as e:

                print(
                    f"Error page {pg}: {e}"
                )

    return table_chunks


table_chunks = extract_table_chunks(
    PDF_PATH,
    section_ranges
)

print(table_chunks[0])

d:\python scripts\Generative AI\Arise\arise\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (118.66, 353.04472, 538.65192, 487.080400597015)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
d:\python scripts\Generative AI\Arise\arise\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (118.66, 353.04472, 538.65192, 487.080400597015)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
d:\python scripts\Generative AI\Arise\arise\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (118.66, 616.5816, 538.7530399999999, 793.8931927272728)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
d:\python scripts\Generative AI\Arise\arise\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (118.66, 616.5816, 538.7530399999999, 793.8931927272728)
  cols, rows, v_s, h_s = self.

{'section_num': '1', 'title': 'Introduction', 'page_start': 8, 'page_end': 12, 'page_number': 8, 'table_index': 0, 'chunk_type': 'table', 'text': '1.1. Background: 1.1.1. | : The erstwhile Jharkhand State Electricity Board (“Board” or “JSEB”) was a statutory\n1.1. Background:  | : body  constituted  under  Section  5  of  the  Electricity  (Supply)  Act,  1948  and  was\n1.1. Background:  | : engaged in electricity generation, transmission, distribution and related activities in\n1.1. Background:  | : the  State  of  Jharkhand.  The  erstwhile  Jharkhand  State  Electricity  Board  (JSEB)\n1.1. Background:  | : was  constituted  on  March  10,  2001  under  the  Electricity  (Supply)  Act,  1948  as  a\n1.1. Background:  | : result of the bifurcation of the   erstwhile State of Bihar. Before that, the Jharkhand\n1.1. Background:  | : State Electricity Board (JSEB) was the predominant entity entrusted with the task of\n1.1. Background:  | : generating, transmitting and supplying power i

In [ ]:
table_chunks

[{'section_num': '1',
  'title': 'Introduction',
  'page_start': 8,
  'page_end': 12,
  'page_number': 8,
  'table_index': 0,
  'chunk_type': 'table',
  'text': '1.1. Background: 1.1.1. | : The erstwhile Jharkhand State Electricity Board (“Board” or “JSEB”) was a statutory\n1.1. Background:  | : body  constituted  under  Section  5  of  the  Electricity  (Supply)  Act,  1948  and  was\n1.1. Background:  | : engaged in electricity generation, transmission, distribution and related activities in\n1.1. Background:  | : the  State  of  Jharkhand.  The  erstwhile  Jharkhand  State  Electricity  Board  (JSEB)\n1.1. Background:  | : was  constituted  on  March  10,  2001  under  the  Electricity  (Supply)  Act,  1948  as  a\n1.1. Background:  | : result of the bifurcation of the   erstwhile State of Bihar. Before that, the Jharkhand\n1.1. Background:  | : State Electricity Board (JSEB) was the predominant entity entrusted with the task of\n1.1. Background:  | : generating, transmitting and su

## different approach

In [10]:
# !pip install unstructured[pdf]

In [11]:
# from unstructured.partition.pdf import partition_pdf

# elements = partition_pdf(
#     filename=r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf",
#     strategy="hi_res"
# )

# for element in elements:
#     print(type(element).__name__)
#     print(element.text[:100])

In [5]:
# pip install "unstructured[all-docs]"
from unstructured.partition.auto import partition
blocks = partition(filename=r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf")
for block in blocks:
    print(f"{block.category}: {block.text}")

No languages specified, defaulting to English.


Title: BEFORE THE HON’BLE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION
UncategorizedText: FILING OF PETITION FOR PROVISIONAL TRUE UP FOR FY 2023-24,
Title: APR FOR FY 2024-25
Title: AND
Title: ARR AND TARIFF PETITION FOR FY 2025-26
UncategorizedText: SUBMITTEDTO:
Title: JHARKHAND STATE ELECTRICITY REGULATORY
Title: COMMISSION, RANCHI
UncategorizedText: SUBMITTED BY:
Title: JHARKHAND URJA SANCHARAN NIGAM LIMITED, KUSAI COLONY, RANCHI – 834 010
Header: ARR and Tariff Petition for FY 2025-26 JUSNL
Title: BEFORE THE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION, RANCHI
NarrativeText: IN THE MATTER OF: Filing of the Petition for Provisional Truing up for FY 2023-24, APR for FY 2024-25, ARR and Tariff Petition for FY 2025-26 under Jharkhand State Electricity Regulatory Commission (Terms and Conditions for Determination of Transmission Tariff) Regulations, 2020 and its amendments thereof and directives issued by the JSERC from time to time and under Section 61, 62, 64 and 86 of The E

In [ ]:
for block in blocks:
    print(block.category)
    print(block.text[:100])


Title
BEFORE THE HON’BLE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION
UncategorizedText
FILING OF PETITION FOR PROVISIONAL TRUE UP FOR FY 2023-24,
Title
APR FOR FY 2024-25
Title
AND
Title
ARR AND TARIFF PETITION FOR FY 2025-26
UncategorizedText
SUBMITTEDTO:
Title
JHARKHAND STATE ELECTRICITY REGULATORY
Title
COMMISSION, RANCHI
UncategorizedText
SUBMITTED BY:
Title
JHARKHAND URJA SANCHARAN NIGAM LIMITED, KUSAI COLONY, RANCHI – 834 010
Header
ARR and Tariff Petition for FY 2025-26 JUSNL
Title
BEFORE THE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION, RANCHI
NarrativeText
IN THE MATTER OF: Filing of the Petition for Provisional Truing up for FY 2023-24, APR for FY 2024-2
Title
AND
NarrativeText
IN THE MATTER OF: Jharkhand Urja Sancharan Nigam Limited (hereinafter to as "JUSNL" or erstwhile “JS
Title
…Petitioner
NarrativeText
The Petitioner respectfully submits as under: -
ListItem
1. The erstwhile Jharkhand State Electricity Board (“Board” or “JSEB”) was a
NarrativeText
statutory b

In [ ]:
# !pip install docling

  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached python_docx-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonref-1.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached antlr4-python3-runtime-4.9.3.tar.gz (

In [14]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

result = converter.convert(r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf")

doc = result.document

[INFO] 2026-06-02 12:47:23,246 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-02 12:47:23,251 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-02 12:47:29,491 [RapidOCR] download_file.py:82: Download size: 4.53MB
[INFO] 2026-06-02 12:47:31,532 [RapidOCR] download_file.py:95: Successfully saved to: D:\python scripts\Generative AI\Arise\arise\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-02 12:47:31,535 [RapidOCR] main.py:57: Using D:\python scripts\Generative AI\Arise\arise\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-02 12:47:31,711 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-02 12:47:31,713 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/cls/ch_ppocr_mobile_v2.0_cls_

DownloadFileException: Failed to download https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/onnx/PP-OCRv4/cls/ch_ppocr_mobile_v2.0_cls_mobile.onnx

In [ ]:
markdown = doc.export_to_markdown()

print(markdown[:5000])

%PDF-1.5
%
1 0 obj
<</Type/Catalog/Pages 2 0 R/Lang(en-US) /StructTreeRoot 305 0 R/MarkInfo<</Marked true>>>>
endobj
2 0 obj
<</Type/Pages/Count 66/Kids[ 3 0 R 22 0 R 26 0 R 28 0 R 30 0 R 33 0 R 34 0 R 35 0 R 40 0 R 42 0 R 43 0 R 45 0 R 48 0 R 53 0 R 54 0 R 59 0 R 61 0 R 63 0 R 64 0 R 65 0 R 66 0 R 68 0 R 69 0 R 72 0 R 74 0 R 76 0 R 79 0 R 80 0 R 82 0 R 84 0 R 86 0 R 87 0 R 92 0 R 94 0 R 96 0 R 97 0 R 98 0 R 99 0 R 101 0 R 102 0 R 104 0 R 105 0 R 107 0 R 109 0 R 111 0 R 113 0 R 119 0 R 121 0 R 123 0 R 124 0 R 125 0 R 126 0 R 127 0 R 129 0 R 130 0 R 132 0 R 133 0 R 135 0 R 137 0 R 140 0 R 141 0 R 145 0 R 148 0 R 149 0 R 151 0 R 302 0 R] >>
endobj
3 0 obj
<</Type/Page/Parent 2 0 R/Resources<</Font<</F1 5 0 R/F2 9 0 R/F3 14 0 R/F4 16 0 R>>/ExtGState<</GS7 7 0 R/GS8 8 0 R>>/XObject<</Image21 21 0 R>>/ProcSet[/PDF/Text/ImageB/ImageC/ImageI] >>/MediaBox[ 0 0 596.04 842.04] /Contents 4 0 R/Group<</Type/Group/S/Transparency/CS/DeviceRGB>>/Tabs/S/StructParents 0>>
endobj
4 0 obj
<</


In [ ]:
# !pip install pymupdf4llm

   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ------- -------------------------------- 3.7/19.2 MB 16.8 MB/s eta 0:00:01
   --------- ------------------------------ 4.7/19.2 MB 11.0 MB/s eta 0:00:02
   ---------- ----------------------------- 5.0/19.2 MB 8.9 MB/s eta 0:00:02
   ----------- ---------------------------- 5.8/19.2 MB 6.8 MB/s eta 0:00:02
   -------------- ------------------------- 6.8/19.2 MB 6.4 MB/s eta 0:00:02
   ---------------- ----------------------- 7.9/19.2 MB 6.1 MB/s eta 0:00:02
   ----------------- ---------------------- 8.7/19.2 MB 5.9 MB/s eta 0:00:02
   -------------------- ------------------- 9.7/19.2 MB 5.8 MB/s eta 0:00:02
   ---------------------- ----------------- 11.0/19.2 MB 5.8 MB/s eta 0:00:02
   ------------------------- -------------- 12.1/19.2 MB 5.7 MB/s eta 0:00:02
   --------------------------- ------------ 13.1/19.2 MB 5.6 MB/s eta 0:00:02
   ---------------------------- ----------- 13.9/19.2 MB 5.5 MB/s eta 0:00:01


In [15]:
import pymupdf4llm

markdown = pymupdf4llm.to_markdown(
    "D:\\python scripts\\Generative AI\\arise_chatbot\\data\\petitions\\tariff_petition_jusnl.pdf"
)








In [ ]:
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.schema import Document

doc = Document(text=markdown)

parser = MarkdownNodeParser()

nodes = parser.get_nodes_from_documents([doc])

[TextNode(id_='9df68a7c-02ef-44db-bc83-58b30f4d5262', embedding=None, metadata={'header_path': '/'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='89f64c2d-9a89-4749-a961-b4332131de95', node_type=<ObjectType.DOCUMENT: '4'>, metadata={}, hash='399fca9c347517862a0ec0549b805bd61ee2edb10e51bbb3b0ced575334bf0bf'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='313c0728-a510-4026-bd83-816d65e56a48', node_type=<ObjectType.TEXT: '1'>, metadata={'header_path': '/'}, hash='d93ddc115240b41b3f08b592cfa3982b6a718d892d55c426f6841e3ac2122432')}, metadata_template='{key}: {value}', metadata_separator='\n', text='## **BEFORE THE HON’BLE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION** \n\n**==> picture [168 x 169] intentionally omitted <==**\n\n**FILING OF PETITION FOR PROVISIONAL TRUE UP FOR FY 2023-24, APR FOR FY 2024-25** \n\n**AND** \n\n**ARR AND TARIFF PETITION FOR FY 2025-26** \n\nSUBMITTEDTO: \n\

In [19]:
for node in nodes[:5]:
    print("=" * 50)
    print(node.text[:500])

## **BEFORE THE HON’BLE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION** 

**==> picture [168 x 169] intentionally omitted <==**

**FILING OF PETITION FOR PROVISIONAL TRUE UP FOR FY 2023-24, APR FOR FY 2024-25** 

**AND** 

**ARR AND TARIFF PETITION FOR FY 2025-26** 

SUBMITTEDTO: 

**JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION, RANCHI** 

SUBMITTED BY: 

**JHARKHAND URJA SANCHARAN NIGAM LIMITED,** KUSAI COLONY, RANCHI – 834 010 

ARR and Tariff Petition for FY 2025-26                  
## **BEFORE THE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION, RANCHI** 

- IN THE MATTER OF: Filing of the Petition for Provisional Truing up for FY 2023-24, APR for FY 2024-25, ARR and Tariff Petition for FY 2025-26 under Jharkhand State Electricity Regulatory Commission (Terms and Conditions for Determination of Transmission Tariff) Regulations, 2020 and its amendments thereof and directives issued by the JSERC from time to time and under Section 61, 62, 64 and 86 of The Electricity Act 

In [13]:
from llama_index.core.schema import Document
from llama_index.core.node_parser import MarkdownNodeParser

document = Document(text=markdown)

parser = MarkdownNodeParser()

nodes = parser.get_nodes_from_documents(
    [document]
)

In [15]:
sections = []

for node in nodes:
    sections.append({
        "title": node.metadata,
        "content": node.text
    })

In [ ]:
import pymupdf4llm

markdown = pymupdf4llm.to_markdown(
    r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf"
)

with open("output.md", "w", encoding="utf-8") as f:
    f.write(markdown)

from llama_index.core import Document

document = Document(
    text=markdown
)

# from llama_index.core.node_parser import MarkdownNodeParser

# parser = MarkdownNodeParser()

# nodes = parser.get_nodes_from_documents(
#     [document]
# )

# for node in nodes:

#     print("=" * 100)

#     print("METADATA:")
#     print(node.metadata)

#     print()

#     print("TEXT:")
#     print(node.text[:500])


'## **BEFORE THE HON’BLE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION**\n\n**==> picture [168 x 169] intentionally omitted <==**\n\n**FILING OF PETITION FOR PROVISIONAL TRUE UP FOR FY 2023-24, APR FOR FY 2024-25**\n\n**AND**\n\n**ARR AND TARIFF PETITION FOR FY 2025-26**\n\nSUBMITTEDTO:\n\n**JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION, RANCHI**\n\nSUBMITTED BY:\n\n**JHARKHAND URJA SANCHARAN NIGAM LIMITED,** KUSAI COLONY, RANCHI – 834 010\n\nARR and Tariff Petition for FY 2025-26                                                                                  JUSNL\n\n## **BEFORE THE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION, RANCHI**\n\n- IN THE MATTER OF: Filing of the Petition for Provisional Truing up for FY 2023-24, APR for FY 2024-25, ARR and Tariff Petition for FY 2025-26 under Jharkhand State Electricity Regulatory Commission (Terms and Conditions for Determination of Transmission Tariff) Regulations, 2020 and its amendments thereof and directives issued by the

In [32]:

import re

new_lines = []

for line in markdown.split("\n"):

    line = line.strip()

    # Main Section
    if re.match(r"^##\s*\*\*\d+\.\s", line):
        line = "# " + re.sub(r"^##\s*\*\*", "", line).replace("**", "")

    # Subsection
    elif re.match(r"^##\s*\*\*\d+\.\d+\.", line):
        line = "## " + re.sub(r"^##\s*\*\*", "", line).replace("**", "")

    # Sub-subsection
    elif re.match(r"^##\s*\*\*\d+\.\d+\.\d+\.", line):
        line = "### " + re.sub(r"^##\s*\*\*", "", line).replace("**", "")

    new_lines.append(line)

markdown = "\n".join(new_lines)

In [33]:
from llama_index.core import Document
from llama_index.core.node_parser import MarkdownNodeParser

doc = Document(text=markdown)

parser = MarkdownNodeParser()

nodes = parser.get_nodes_from_documents([doc])

for node in nodes[:20]:

    print(node.metadata)

    print(node.text[:300])

    print("-"*80)

{'header_path': '/'}
## **BEFORE THE HON’BLE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION**

**==> picture [168 x 169] intentionally omitted <==**

**FILING OF PETITION FOR PROVISIONAL TRUE UP FOR FY 2023-24, APR FOR FY 2024-25**

**AND**

**ARR AND TARIFF PETITION FOR FY 2025-26**

SUBMITTEDTO:

**JHARKHAND STATE 
--------------------------------------------------------------------------------
{'header_path': '/'}
## **BEFORE THE JHARKHAND STATE ELECTRICITY REGULATORY COMMISSION, RANCHI**

- IN THE MATTER OF: Filing of the Petition for Provisional Truing up for FY 2023-24, APR for FY 2024-25, ARR and Tariff Petition for FY 2025-26 under Jharkhand State Electricity Regulatory Commission (Terms and Conditions f
--------------------------------------------------------------------------------
{'header_path': '/'}
## **Prayers before the Hon’ble Commission:**

The Petitioner respectfully prays that the Hon’ble Commission may:

- a. Admit the instant Petition;

- b. Examine the proposa